# 01 - Groq com Python

**Groq** é uma plataforma de inferência de LLMs extremamente rápida, baseada em hardware proprietário (LPU — Language Processing Unit).
Ela hospeda modelos open-source como LLaMA, Mixtral e Whisper, oferecendo velocidades muito superiores às APIs tradicionais.

```
Sua aplicação  ──►  API Groq  ──►  LPU (hardware dedicado)  ──►  resposta em milissegundos
```

### Principais modelos disponíveis

| Modelo | Tipo | Uso recomendado |
|--------|------|-----------------|
| `llama-3.3-70b-versatile` | Chat | Uso geral, alta qualidade |
| `llama3-8b-8192` | Chat | Rápido e leve |
| `mixtral-8x7b-32768` | Chat | Contexto longo (32k tokens) |
| `whisper-large-v3` | Áudio | Transcrição de voz |
| `llama-3.2-11b-vision-preview` | Visão | Análise de imagens |

### Pacotes necessários

```bash
pip install groq python-dotenv
```

> **Chave de API**: crie uma conta em [console.groq.com](https://console.groq.com) e gere uma API key gratuita.
> Adicione `GROQ_API_KEY=sua_chave` no arquivo `.env`.

In [7]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

## 1. Chat básico

A API do Groq segue o mesmo padrão da OpenAI — `client.chat.completions.create()` —
o que facilita a migração entre provedores.

| Parâmetro | Descrição |
|-----------|----------|
| `messages` | Lista de mensagens com `role` e `content` |
| `model` | Modelo a usar (ver tabela acima) |
| `temperature` | Criatividade: 0 = determinístico, 2 = criativo |
| `stream` | `True` para receber tokens progressivamente |

In [8]:
from groq import Groq

# O cliente lê GROQ_API_KEY automaticamente do ambiente
client = Groq()

mensagens = [
    {
        "role": "user",
        "content": "Explique a diferença entre LLM e LangChain. Responda em português."
    }
]

response = client.chat.completions.create(
    messages=mensagens,
    model="llama-3.3-70b-versatile"  # modelo atualizado — llama3-8b-8192 ainda funciona
)

print(response.choices[0].message.content)

**Introdução**

Com o avanço da inteligência artificial e o desenvolvimento de modelos de linguagem, surgiram duas tecnologias importantes: LLM (Modelos de Linguagem Grande) e LangChain. Embora ambas estejam relacionadas à processamento de linguagem natural, elas têm objetivos e características diferentes. Neste artigo, vamos explorar as diferenças entre LLM e LangChain.

**LLM (Modelos de Linguagem Grande)**

Os Modelos de Linguagem Grande (LLM) são uma classe de modelos de inteligência artificial projetados para processar e gerar texto de forma natural. Eles são treinados em grandes conjuntos de dados textuais e utilizam técnicas de aprendizado de máquina para aprender padrões e relações na linguagem. Os LLMs são capazes de realizar tarefas como:

* Geração de texto
* Resumo de textos
* Tradução de textos
* Resposta a perguntas

Os LLMs são geralmente baseados em arquiteturas de rede neural profunda, como as Redes Neurais Recorrentes (RNN) e as Redes Neurais Convolucionais (CNN). Exe

## 2. Streaming

Com `stream=True`, os tokens chegam progressivamente — ideal para interfaces de chat
onde o usuário vê a resposta sendo gerada em tempo real.

> **`flush=True`**: força o Python a imprimir imediatamente sem buffer,
> garantindo que cada token apareça assim que chega.

In [9]:
stream = client.chat.completions.create(
    messages=mensagens,
    model="llama-3.3-70b-versatile",
    stream=True  # ativa o streaming
)

# Cada chunk contém um fragmento do texto gerado
for chunk in stream:
    conteudo = chunk.choices[0].delta.content
    if conteudo:  # o último chunk pode ser None
        print(conteudo, end="", flush=True)

**Introdução**

Nos últimos anos, o desenvolvimento de modelos de linguagem e suas aplicações tem sido um tópico quente na área de inteligência artificial. Dois conceitos que têm ganhado atenção são LLM (Large Language Model) e LangChain. Embora ambos estejam relacionados à processamento de linguagem natural, há diferenças importantes entre eles.

**LLM (Large Language Model)**

Um LLM é um tipo de modelo de linguagem que utiliza redes neurais para processar e gerar texto. Esses modelos são treinados em grandes conjuntos de dados e podem aprender a representar a estrutura e o significado da linguagem. LLMs são capazes de realizar tarefas como:

* Geração de texto
* Tradução automática
* Resumo de texto
* Análise de sentimento

Exemplos de LLMs incluem o BERT, o RoBERTa e o Transformer.

**LangChain**

LangChain é um framework que permite a construção de aplicações de processamento de linguagem natural utilizando LLMs. Em outras palavras, LangChain é uma ferramenta que ajuda a criar pro

## 3. Transcrição de Áudio (Whisper)

O Groq hospeda o modelo **Whisper Large V3** da OpenAI, mas com inferência muito mais rápida.
Ideal para transcrever áudios de entrevistas, podcasts, aulas, etc.

| Parâmetro | Descrição |
|-----------|----------|
| `file` | Tupla `(nome_arquivo, bytes)` |
| `model` | `"whisper-large-v3"` |
| `language` | Código ISO do idioma (ex: `"pt"`, `"en"`) |
| `prompt` | Contexto opcional para melhorar a transcrição |

In [10]:
import textwrap


def formatar_texto(texto: str, largura: int = 100) -> None:
    """Formata e imprime texto com quebra de linha automática."""
    print(textwrap.fill(texto, width=largura))


arquivo = "files/curso.mp3"

with open(arquivo, "rb") as audio:
    transcricao = client.audio.transcriptions.create(
        file=(arquivo, audio.read()),
        model="whisper-large-v3",
        response_format="json",
        language="pt",
        # prompt ajuda o modelo a reconhecer termos técnicos do contexto
        prompt="Este é um curso de Hugging Face que usa aplicação com Gradio"
    )

formatar_texto(transcricao.text)

 Seja muito bem-vindo e bem-vinda ao curso Desbravando a IA com Python Explorando modelos de Hugging
Face Neste curso eu quero te mostrar uma das maiores plataformas de inteligência artificial que
possui diversos modelos prontos que é a plataforma do Hugging Face E o meu objetivo neste curso é
explorar dois tipos de modelos Modelos relacionados a processamento de linguagem natural onde
estaremos trabalhando com análise de sentimentos, vamos trabalhar também com informações de
perguntas e respostas, vamos trabalhar também com modelos que nos ajudem a resumir textos e assim
por diante. Por outro lado, nós teremos também modelos relacionados à visão computacional. Estaremos
trabalhando com aplicações para segmentar imagens, classificar imagens, detectar objetos e muito
mais. Space além disso em algumas sessões você vai ter a oportunidade de construir o teu modelo é
uma interface web para o teu modelo utilizando a biblioteca Gradio e também vamos aprender a colocar
a aplicação que construi

## 4. Visão (Vision)

O Groq suporta modelos multimodais que recebem **imagens e texto** na mesma mensagem.
A imagem pode ser passada como URL pública ou como base64.

> **Modelos de visão disponíveis**: `llama-3.2-11b-vision-preview` e `llama-3.2-90b-vision-preview`.
> São modelos preview — funcionalidades podem mudar.

In [14]:
completion = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{
        "role": "user",
        "content": [
            # texto e imagem podem ser combinados na mesma mensagem
            {"type": "text", "text": "O que há nessa imagem?"},
            {"type": "image_url", "image_url": {
                "url": "https://www.civitatis.com/blog/wp-content/uploads/2023/12/shutterstock_318248558-scaled.jpg"
            }}
        ]
    }],
    temperature=1,
    max_completion_tokens=1024,
)

formatar_texto(completion.choices[0].message.content)

Nessa imagem, vemos uma foto da famosa praia de Copacabana. No primeiro plano temos palmeiras, uma
área de calçada, e a areia da praia. Ao fundo podemos ver o mar e o Morro do Arpoador.


In [13]:
# Lista todos os modelos disponíveis na sua conta Groq
models = client.models.list()

for model in models.data:
    print(model.id)

canopylabs/orpheus-v1-english
allam-2-7b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
whisper-large-v3
whisper-large-v3-turbo
groq/compound
openai/gpt-oss-20b
llama-3.3-70b-versatile
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct
canopylabs/orpheus-arabic-saudi
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
llama-3.1-8b-instant


## Resumo

| Funcionalidade | Método | Modelo |
|---------------|--------|--------|
| Chat | `client.chat.completions.create()` | `llama-3.3-70b-versatile` |
| Streaming | `stream=True` + loop no iterador | qualquer modelo de chat |
| Transcrição de áudio | `client.audio.transcriptions.create()` | `whisper-large-v3` |
| Análise de imagem | `content` com `image_url` | `llama-3.2-11b-vision-preview` |

> **Próximo passo**: usar o Groq com LangChain via `ChatGroq` para aproveitar
> toda a velocidade do Groq dentro do ecossistema LangChain (chains, agentes, RAG).